# GSAS-II Rietveld refinement

Pick a diffraction pattern and one or more phase CIFs, set the fit window, and
run. The tool reports **lattice parameters, crystallite size, microstrain and
Rₓₚ**, draws an obs/calc/difference plot, and writes `results.csv`.

*Crystallite size and microstrain are **relative** until a LaB₆/Si standard is
measured and its profile terms go into the `.instprm` file.*


In [ ]:
import os, glob, csv
import matplotlib
matplotlib.use("Agg")           # headless: save PNGs, no display server needed
import ipywidgets as w
from IPython.display import display, Image
import rietveld_batch as rb     # the refinement engine (ships beside this notebook)

try:
    import pandas as pd
except Exception:
    pd = None

# Import GSAS-II once. If it is not installed in this image/session, the app
# still renders and tells the user how to fix it instead of crashing.
try:
    G2sc = rb._import_g2(); G2_OK = True; G2_ERR = ""
except Exception as e:
    G2sc = None; G2_OK = False; G2_ERR = str(e)

PATTERN_EXT = ("*.raw", "*.xy", "*.xye", "*.dat", "*.txt", "*.ascii", "*.chi")
ST = {"description_width": "110px"}
WIDE = w.Layout(width="60%")

folder   = w.Text(value=".", description="Data folder:", layout=WIDE, style=ST)
scan_btn = w.Button(description="Scan folder", icon="search")
pat_sel  = w.SelectMultiple(description="Pattern(s):", rows=6, layout=WIDE, style=ST)
cif_sel  = w.SelectMultiple(description="Phase CIF(s):", rows=6, layout=WIDE, style=ST)
ins_sel  = w.Dropdown(description="Instrument:", layout=WIDE, style=ST)
lo   = w.FloatText(value=15.0, description="2θ min:", style=ST, layout=w.Layout(width="170px"))
hi   = w.FloatText(value=90.0, description="2θ max:", style=ST, layout=w.Layout(width="170px"))
bkg  = w.IntText(value=8, description="Bkg terms:", style=ST, layout=w.Layout(width="170px"))
size_cb   = w.Checkbox(value=True, description="Refine crystallite size")
strain_cb = w.Checkbox(value=True, description="Refine microstrain")
outdir = w.Text(value="rietveld_out", description="Output dir:", layout=WIDE, style=ST)
run_btn = w.Button(description="Run refinement", button_style="primary", icon="play")
out = w.Output()

def scan(_=None):
    d = folder.value or "."
    pats = []
    for e in PATTERN_EXT:
        pats += glob.glob(os.path.join(d, e))
    pat_sel.options = sorted({os.path.basename(p) for p in pats})
    cif_sel.options = sorted(os.path.basename(p) for p in glob.glob(os.path.join(d, "*.cif")))
    instr = sorted(os.path.basename(p) for p in glob.glob(os.path.join(d, "*.instprm")))
    ins_sel.options = (["(none / default)"] + instr) if instr else ["(none / default)"]
scan_btn.on_click(scan)

def run_refine(_=None):
    out.clear_output()
    with out:
        if not G2_OK:
            print("GSAS-II is not available in this environment:\n ", G2_ERR); return
        d = folder.value or "."
        patterns = list(pat_sel.value)
        cifs = [os.path.join(d, c) for c in cif_sel.value]
        if not patterns:
            print("Pick at least one pattern."); return
        if not cifs:
            print("Pick at least one phase CIF."); return
        instprm = (os.path.join(d, ins_sel.value)
                   if ins_sel.value and not ins_sel.value.startswith("(none") else None)
        od = outdir.value or "rietveld_out"; os.makedirs(od, exist_ok=True)
        cfg = dict(rb.CONFIG)
        cfg.update(twotheta_limits=(lo.value, hi.value), background_order=int(bkg.value),
                   refine_size=size_cb.value, refine_strain=strain_cb.value, out_dir=od)
        rows = []
        for patt in patterns:
            name = os.path.splitext(patt)[0]
            print(f"Refining {patt} ...")
            try:
                gpx = G2sc.G2Project(newgpx=os.path.join(od, name + ".gpx"))
                hist = rb.load_histogram(gpx, os.path.join(d, patt), instprm, od)
                phases = [gpx.add_phase(c, histograms=[hist], fmthint="CIF") for c in cifs]
                rb.refine_staged(gpx, hist, phases, cfg); gpx.save()
                rwp = hist.get_wR()
                wts = rb.weight_fractions(phases, hist) if len(phases) > 1 else {phases[0].name: 100.0}
                for ph in phases:
                    r = dict(file=name, phase=ph.name, Rwp=round(rwp, 3),
                             wt_pct=round(wts.get(ph.name, float("nan")), 2))
                    r.update(rb.extract(ph, hist)); rows.append(r)
                rb.plot_fit(hist, phases, os.path.join(od, name + "_fit.png"),
                            f"{name}  Rwp={rwp:.2f}%", tth_range=(lo.value, hi.value))
                rb.export_profile(hist, os.path.join(od, name + "_profile.csv"))
                print(f"   done: Rwp={rwp:.2f}%")
            except Exception as e:
                print(f"   FAILED: {e}")
        if rows:
            cols = ["file", "phase", "Rwp", "wt_pct", "a", "a_esd", "vol", "size_nm", "microstrain_pct"]
            with open(os.path.join(od, "results.csv"), "w", newline="") as fh:
                wr = csv.DictWriter(fh, fieldnames=cols, extrasaction="ignore"); wr.writeheader()
                for r in rows: wr.writerow(r)
            if pd is not None:
                display(pd.DataFrame(rows)[[c for c in cols if any(c in r for r in rows)]])
            else:
                for r in rows: print(r)
            for patt in patterns:
                png = os.path.join(od, os.path.splitext(patt)[0] + "_fit.png")
                if os.path.exists(png): display(Image(filename=png))
            print(f"\nSaved results.csv + fit plots to: {od}")
            print("NOTE: crystallite size & microstrain are RELATIVE until a LaB6/Si "
                  "standard is measured and its profile terms go into the .instprm file.")
run_btn.on_click(run_refine)

banner = ("" if G2_OK else
          "<div style='background:#fdecea;border:1px solid #f5c6cb;padding:8px;border-radius:4px'>"
          "<b>GSAS-II not found in this environment.</b> Install it in the tool image "
          "(see the Dockerfile) — the refinement button will not work until then.</div>")

display(w.VBox([
    w.HTML("<h2>GSAS-II Rietveld refinement</h2>"
           "<p>Pick a pattern and one or more phase CIFs, then run. Outputs: lattice "
           "parameters, crystallite size, microstrain, R<sub>wp</sub> and a fit plot.</p>" + banner),
    w.HBox([folder, scan_btn]),
    pat_sel, cif_sel, ins_sel,
    w.HBox([lo, hi, bkg]),
    w.HBox([size_cb, strain_cb]),
    outdir, run_btn, out,
]))
scan()   # auto-scan the default folder on load